In [2]:
from openai import OpenAI
import pandas as pd
import json
import duckdb
from pydantic import BaseModel, Field
from IPython.display import Markdown
#from helper import get_openai_api_key
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [4]:
# initialize the OpenAI client
#openai_api_key = get_openai_api_key()
client = OpenAI()

MODEL = "gpt-4o-mini"

In [5]:
# define the path to the transactional data
TRANSACTION_DATA_FILE_PATH = 'synthetic_users.parquet'

In [17]:
import pandas as pd
pd.read_parquet('synthetic_users.parquet',engine='pyarrow')

,user_id,name,email,address,phone_number,age,gender,salary
0,1,Brian Butler,grant07@example.org,"21999 Walker Fork, South Gabrielhaven, WY 74371",698.295.9797x3138,45,Female,71499.89
1,2,Scott Aguirre,smurphy@example.net,"458 Jackson Place, South Ruthmouth, DC 34246",001-213-918-8317x83481,29,Male,135758.12
2,3,Rachel Cox,byrdcharles@example.net,"USS Banks, FPO AE 73423",(269)454-9447,28,Female,91726.43
3,4,Mary Snyder,stephenrodriguez@example.org,"9241 Marc Harbor, North Marymouth, AS 16590",307.440.8830x59998,25,Female,79148.98
4,5,Daniel Washington,emilyjohnson@example.com,"48692 Wells Landing, New Margaret, DC 21638",(208)799-3839x902,18,Female,127840.92
...,...,...,...,...,...,...,...,...
95,96,Peter Alexander,gweiss@example.net,"03506 Stephanie Forest, Archerton, MI 50140",339-829-1563,55,Male,114707.44
96,97,Elizabeth Hammond,ballardsuzanne@example.net,"577 Spencer Dale, North Staceyview, OH 47637",+1-650-694-7404x560,44,Other,129890.80
97,98,Adam Romero,asmall@example.com,"6896 Morris Knolls, Jimmystad, ME 71365",838-373-3312,19,Male,145958.34
98,99,Amber Smith,sedwards@example.net,"7156 Bradley Valley Apt. 171, North Joseph, OK...",(433)664-5947,32,Male,140844.65


In [6]:
# prompt template for step 2 of tool 1
SQL_GENERATION_PROMPT = """
Generate an SQL query based on a prompt. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""

In [7]:
# code for step 2 of tool 1
def generate_sql_query(prompt: str, columns: list, table_name: str) -> str:
    """Generate an SQL query based on a prompt"""
    formatted_prompt = SQL_GENERATION_PROMPT.format(prompt=prompt, 
                                                    columns=columns, 
                                                    table_name=table_name)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    return response.choices[0].message.content

In [8]:
# code for tool 1
def lookup_sales_data(prompt: str) -> str:
    """Implementation of sales data lookup from parquet file using SQL"""
    try:

        # define the table name
        table_name = "sales"
        
        # step 1: read the parquet file into a DuckDB table
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")

        # step 2: generate the SQL code
        sql_query = generate_sql_query(prompt, df.columns, table_name)
        # clean the response to make sure it only includes the SQL code
        sql_query = sql_query.strip()
        sql_query = sql_query.replace("```sql", "").replace("```", "")
        
        # step 3: execute the SQL query
        result = duckdb.sql(sql_query).df()
        
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {str(e)}"

In [12]:
example_data = lookup_sales_data("Display only the name of all users who are older than 60 years")
print(example_data)

    user_id
0         2
1         6
2         9
3        24
4        25
5        29
6        30
7        47
8        52
9        59
10       60
11       62
12       68
13       70
14       71
15       72
16       76


In [ ]:
# code for tool 1
def lookup_sales_data(prompt: str) -> str:
    """Implementation of sales data lookup from parquet file using SQL"""
    try:

        # define the table name
        table_name = "sales"
        
        # step 1: read the parquet file into a DuckDB table
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")

        # step 2: generate the SQL code
        sql_query = generate_sql_query(prompt, df.columns, table_name)
        # clean the response to make sure it only includes the SQL code
        sql_query = sql_query.strip()
        sql_query = sql_query.replace("```sql", "").replace("```", "")
        
        # step 3: execute the SQL query
        result = duckdb.sql(sql_query).df()
        
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {str(e)}"

#### Data Analysis

In [18]:
# Construct prompt based on analysis type and data subset
DATA_ANALYSIS_PROMPT = """
Analyze the following data: {data}
Your job is to answer the following question: {prompt}
"""

In [19]:
# code for tool 2
def analyze_sales_data(prompt: str, data: str) -> str:
    """Implementation of AI-powered sales data analysis"""
    formatted_prompt = DATA_ANALYSIS_PROMPT.format(data=data, prompt=prompt)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    analysis = response.choices[0].message.content
    return analysis if analysis else "No analysis could be generated"

In [20]:
print(analyze_sales_data(prompt="what trends do you see in this data", 
                         data=example_data))

To analyze the provided user IDs, we can look for trends and characteristics in the data, even though it appears to be a relatively small sample. Here’s an analysis based on the user IDs listed:

1. **Range and Distribution**: 
   - The user IDs range from 2 to 76.
   - The IDs are not continuous; there are gaps, indicating that not all IDs in this range are being used.

2. **Frequency of IDs**: 
   - Some IDs are notably missing. For instance, user IDs from 1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, and others are absent.
   - The largest gap in the sequence is between 24 and 25, suggesting that multiple users or entities might have been skipped in the assignment of user IDs.

3. **Clustering of IDs**: 
   - There’s a notable gap after user_id 9 until user_id 24, suggesting a potential cluster of users in the lower range of IDs and a transition to a higher range thereafter.
   - The IDs seem to be relatively sparse, especially at the lower end (between IDs 1 and 10). 

4. **Poss

#### Data Visualization

In [ ]:
# prompt template for step 1 of tool 3
CHART_CONFIGURATION_PROMPT = """
Generate a chart configuration based on this data: {data}
The goal is to show: {visualization_goal}
"""